# Clase 6 — Módulos y bibliotecas estándar: organizar y reutilizar tu código

**Curso:** Python sin miedo — UNSL, FCFMyN · **Módulo 3, semana 6**

## Objetivos de hoy

1. Dominar el **`import`** en todas sus variantes.
2. Recorrer cuatro bibliotecas estándar que vas a usar siempre: **`math`**, **`random`**, **`datetime`** y **`os`**/**`pathlib`**.
3. Crear **tus propios módulos**: archivos `.py` con tus funciones y clases, importables desde cualquier lado.
4. Organizar el código de un proyecto (con la mira puesta en tu proyecto final).

Python viene con *baterías incluidas*: una **biblioteca estándar** de decenas de módulos listos para usar, sin instalar nada. Ya usamos varios sin presentarlos formalmente (`csv`, `json`, `datetime`). Hoy los recorremos con método — y aprendemos a fabricar los nuestros.

## 1. `import` en todas sus formas

| Forma | Cuándo conviene |
|---|---|
| `import math` | la general: se usa como `math.sqrt(2)` — siempre se sabe de dónde vino cada cosa |
| `from math import sqrt, pi` | cuando usás pocas cosas muchas veces: `sqrt(2)` |
| `import numpy as np` | el alias: estándar para nombres largos (lo vas a ver en el módulo 4) |
| `from math import *` | ⚠️ **evitala**: llena tu espacio de nombres de cosas que no sabés que importaste |

In [ ]:
import math
print(math.sqrt(2))

from math import pi, sqrt
print(sqrt(2) * pi)

# El import se hace UNA vez por sesión (Python lo cachea):
# repetirlo no cuesta nada, y la convención es ponerlos todos al inicio del archivo.

## 2. `math`: la calculadora científica

Lo esencial: `sqrt`, `exp`, `log` (natural), `log10`, `sin`/`cos`/`tan` (¡en **radianes**!), las constantes `pi` y `e`, y los redondeos `floor`/`ceil`.

In [ ]:
import math

# Decaimiento exponencial: N(t) = N0 * e^(-lambda*t)
n0 = 1000
lam = 0.05
for t in [0, 10, 20, 40]:
    print(f't={t:>3}: N = {n0 * math.exp(-lam * t):8.2f}')

print()
print('seno de 30°:', math.sin(math.radians(30)))   # grados → radianes primero
print('log10(1000):', math.log10(1000))
print('ln(e²):     ', math.log(math.e ** 2))

Un clásico que tarde o temprano muerde: los floats tienen redondeo binario, así que **nunca compares floats con `==`**. Para eso existe `math.isclose`:

In [ ]:
print(0.1 + 0.2 == 0.3)                  # ¡False! bienvenidos al punto flotante
print(math.isclose(0.1 + 0.2, 0.3))      # True: así se compara bien

### 🖊️ Tu turno 1

Ley de enfriamiento de Newton: $T(t) = T_{amb} + (T_0 - T_{amb})\,e^{-kt}$. Para un mate que arranca a $T_0 = 90$ °C en un ambiente a $T_{amb} = 20$ °C con $k = 0.1$ por minuto: calculá la temperatura a los 10 minutos, y buscá con un bucle el primer minuto entero en que baja de 30 °C.

Verificación: T(10) ≈ 45.75 °C; baja de 30 °C en el minuto 20.

In [ ]:
# Tu código acá


## 3. `random`: azar... reproducible

Sortear, submuestrear, simular ruido de medición, métodos Monte Carlo: la ciencia usa el azar todo el tiempo. El módulo `random` trae:

In [ ]:
import random

print(random.random())            # float uniforme en [0, 1)
print(random.uniform(-1, 1))      # float uniforme en un rango
print(random.randint(1, 6))       # entero (incluye ambos extremos): un dado
print(random.gauss(20, 2))        # normal con media 20 y desvío 2: ruido de medición

sitios = ['Norte', 'Sur', 'Este', 'Oeste']
print(random.choice(sitios))      # elegir uno
print(random.sample(sitios, 2))   # elegir k SIN reposición (submuestreo)
# random.shuffle(lista) mezcla la lista en el lugar (aleatorizar un orden de ensayos)

Ejecutá esa celda dos veces: da distinto cada vez. Para un script de análisis eso es un problema serio — tus resultados no se pueden **reproducir** (ni por tus colegas, ni por vos en seis meses, ni por el revisor 2). La solución es fijar la **semilla** del generador:

In [ ]:
random.seed(42)        # a partir de acá, la secuencia "aleatoria" es siempre la misma
print([random.randint(1, 6) for _ in range(5)])

random.seed(42)        # misma semilla → misma secuencia, en cualquier máquina
print([random.randint(1, 6) for _ in range(5)])

# Regla de oro científica: toda simulación publica su semilla.

### 🖊️ Tu turno 2

Con `random.seed(123)`, simulá 30 tiradas de un dado y contá la frecuencia de cada cara con el patrón de diccionario contador del módulo 2. ¿Salió equilibrado? (Con 30 tiradas, no esperes perfección.)

In [ ]:
# Tu código acá


## 4. `datetime`: fechas y tiempos sin sufrir

Calcular días entre campañas, marcar cuándo se generó un resultado, interpretar fechas de un archivo: todo eso es `datetime`. Las piezas: `date` (fecha), `datetime` (fecha y hora) y `timedelta` (duración).

In [ ]:
from datetime import date, datetime, timedelta

hoy = date.today()
print('Hoy:', hoy)
print('Ahora:', datetime.now())

# Aritmética de fechas: restar fechas da un timedelta
entrega_proyecto = date(2026, 11, 15)
faltan = entrega_proyecto - hoy
print(f'Faltan {faltan.days} días para la presentación del proyecto final')

# Sumar duraciones
print('En 3 semanas:', hoy + timedelta(weeks=3))

In [ ]:
# Texto → fecha (strptime: "parse") y fecha → texto (strftime: "format")
campania = datetime.strptime('14/09/2026', '%d/%m/%Y')
print(campania)

print(campania.strftime('%Y-%m-%d'))          # ideal para nombres de archivo: ordena solo
print(datetime.now().strftime('%d/%m/%Y %H:%M'))

# Los códigos: %d día, %m mes, %Y año de 4 cifras, %H:%M:%S hora. Hay tabla completa en la docu.

## 5. `os` y `pathlib`: hablar con el sistema de archivos

Para scripts que procesan *muchos* archivos (toda una carpeta de salidas de instrumento, por ejemplo) hay que poder listar, crear carpetas y armar rutas. La forma moderna es `pathlib`; vas a ver mucho `os` en código ajeno, así que mostramos los dos:

In [ ]:
import os
from pathlib import Path

print('Carpeta de trabajo:', os.getcwd())     # equivalente: Path.cwd()

# Crear una carpeta (sin error si ya existe)
Path('salidas').mkdir(exist_ok=True)

# Armar rutas con / (pathlib une con la barra correcta en Windows, Linux y Mac)
ruta = Path('salidas') / 'prueba.txt'
with open(ruta, 'w', encoding='utf-8') as f:
    f.write('hola desde pathlib\n')

print('¿Existe?', ruta.exists())
print('Archivos .txt en salidas/:', list(Path('salidas').glob('*.txt')))

La estrella es `.glob('*.txt')`: busca por patrón, como en el explorador. Patrón típico de procesamiento por lotes:

```python
for archivo in Path('datos').glob('*.csv'):
    procesar(archivo)        # ¡el bucle de los 200 archivos del instrumento!
```

⚠️ Nunca escribas rutas absolutas de tu máquina (`C:\Users\vos\...`) en el código: el script muere en cualquier otra computadora. Usá rutas relativas a la carpeta del proyecto, como `datos/algo.csv`.

## 6. Tu propio módulo: de celdas sueltas a herramienta

Un **módulo** no tiene nada de mágico: es un archivo `.py`. Si tus funciones viven en `analisis.py`, cualquier script o notebook *en la misma carpeta* puede hacer `import analisis`. Se acabó el copiar y pegar celdas entre notebooks.

Para crear el archivo desde el notebook usamos lo aprendido en la clase 4 (en VS Code simplemente crearías el archivo nuevo y listo):

In [ ]:
codigo_modulo = '''
"""analisis.py — Herramientas de análisis del curso Python sin miedo."""

def media(valores):
    """Media aritmética. Lanza ValueError si la lista está vacía."""
    if not valores:
        raise ValueError('Lista vacía')
    return sum(valores) / len(valores)


def desviacion_estandar(valores, muestral=True):
    """Desvío estándar (divide por n-1 si muestral)."""
    n = len(valores)
    if n < 2:
        raise ValueError('Se necesitan al menos 2 valores')
    m = media(valores)
    divisor = n - 1 if muestral else n
    return (sum((x - m) ** 2 for x in valores) / divisor) ** 0.5


class Muestra:
    """Muestra de laboratorio con pH validado."""

    def __init__(self, codigo, sitio, ph):
        if not 0 <= ph <= 14:
            raise ValueError(f'pH fuera de rango: {ph}')
        self.codigo = codigo
        self.sitio = sitio
        self.ph = ph

    def __repr__(self):
        return f'Muestra({self.codigo!r}, {self.sitio!r}, ph={self.ph})'

    def es_acida(self):
        return self.ph < 7


if __name__ == '__main__':
    # Esto SOLO corre si ejecutás "python analisis.py" directo:
    # al importarlo desde otro lado, no. Ideal para pruebas rápidas.
    print('Probando analisis.py...')
    print('media:', media([1, 2, 3]))
    print(Muestra('TEST', 'Lab', 7.0))
'''

with open('analisis.py', 'w', encoding='utf-8') as f:
    f.write(codigo_modulo)
print('analisis.py creado ✔')

In [ ]:
import analisis

datos = [2, 4, 4, 4, 5, 5, 7, 9]
print('media:', analisis.media(datos))
print('desvío:', analisis.desviacion_estandar(datos))

m = analisis.Muestra('S-01', 'Norte', 6.8)
print(m, '→ ¿ácida?', m.es_acida())

# También vale: from analisis import media, Muestra

Tres cosas para no tropezar:

1. **El kernel cachea los imports.** Si editás `analisis.py` y volvés a correr `import analisis`, *no* recarga los cambios. Lo simple: `Kernel → Restart & Run All` (otra razón para ese hábito). Lo fino: `import importlib; importlib.reload(analisis)`.
2. **Jamás llames a tu archivo como un módulo estándar.** Un `random.py` o `math.py` tuyo en la carpeta *tapa* al de Python y rompe todo de formas desconcertantes. Clásico absoluto.
3. **`if __name__ == '__main__':`** — dentro de ese bloque va lo que debe ejecutarse solo al correr el archivo directamente (`python analisis.py`), no al importarlo. Así el mismo archivo es biblioteca *y* script de prueba.

## 7. Organizar un proyecto

Con módulos, carpetas y rutas relativas ya podés estructurar un proyecto de verdad — esta es la forma que recomendamos para el **proyecto final**:

```
mi-proyecto/
├── README.md          qué hace, cómo se corre
├── datos/             datos de entrada (crudos, intocables)
├── resultados/        lo que genera el código (regenerable: se puede borrar)
├── analisis.py        tus funciones y clases reutilizables
└── main.py            el script principal (o un notebook) que orquesta todo
```

La regla de oro: **datos crudos separados de resultados**, y todo lo que está en `resultados/` debe poder regenerarse corriendo el código. En el módulo 5 le sumamos Git a esta estructura y queda un proyecto científico reproducible y publicable.

## 8. Práctica integradora (segunda parte de la clase)

### Ejercicio 1 — π por Monte Carlo

El método Monte Carlo más famoso: sorteá `N = 10_000` puntos `(x, y)` con `random.uniform(-1, 1)` para cada coordenada; la fracción que cae dentro del círculo unitario ($x^2 + y^2 \le 1$) multiplicada por 4 estima π. Usá `random.seed(42)`, compará con `math.pi` y calculá el error porcentual.

Verificación: con esa semilla da 3.1392 (error ≈ 0.08 %).

In [ ]:
# Ejercicio 1


### Ejercicio 2 — Módulo de simulación

Creá un archivo `simulacion.py` (con la técnica de la sección 6) que contenga la clase `Particula` de la clase pasada (`masa, x, y, vx, vy`, métodos `paso(dt)`, `energia_cinetica()` y `__repr__`) y una función `simular(particula, pasos, dt)` que avance la partícula y devuelva la lista de posiciones `(x, y)` recorridas. Importalo y simulá 10 pasos de la partícula del otro día (masa 2, velocidad `(3, 4)`, `dt = 0.1`).

In [ ]:
# Ejercicio 2


### Ejercicio 3 — Resultados con fecha

Creá la carpeta `resultados/` con `pathlib` (`exist_ok=True`) y guardá la trayectoria del ejercicio 2 como CSV (columnas `x,y`) en un archivo cuyo nombre incluya la fecha de hoy, tipo `trayectoria_2026-10-20.csv` (pista: `date.today().strftime(...)` o directamente f-string con `date.today()`). Verificá con `.glob('*.csv')` que quedó donde debía.

In [ ]:
# Ejercicio 3


### Ejercicio 4 (desafío) — Caminata browniana

Agregá a `simulacion.py` una clase `ParticulaBrowniana(Particula)` que sobreescriba `paso(dt)`: en lugar de moverse con su velocidad, suma a cada coordenada un ruido `random.gauss(0, 1)`. Con `random.seed(42)`, simulá 1000 pasos desde el origen y reportá la distancia final al origen (`math.hypot(x, y)`).

Verificación: ≈ 26.15. Opcional: repetí la simulación 100 veces (sin resetear la semilla adentro del bucle) y promediá la distancia final — para una caminata aleatoria la teoría predice que crece como $\sqrt{n}$ (√1000 ≈ 31.6).

In [ ]:
# Ejercicio 4


## Resumen de la clase

| Concepto | Sintaxis clave |
|---|---|
| Importar | `import math` / `from math import sqrt` / `import numpy as np` |
| Matemática | `math.sqrt`, `math.exp`, `math.log`, `math.pi`, `math.isclose` (¡no comparar floats con `==`!) |
| Azar | `random.uniform`, `random.randint`, `random.gauss`, `random.choice`, `random.sample` |
| Reproducibilidad | `random.seed(42)` — toda simulación publica su semilla |
| Fechas | `date.today()`, restar fechas → `timedelta`, `strptime` / `strftime` |
| Archivos y carpetas | `Path('datos') / 'a.csv'`, `.mkdir(exist_ok=True)`, `.glob('*.csv')` |
| Módulo propio | un `.py` en la carpeta + `import analisis` |
| Script vs import | `if __name__ == '__main__':` |

## Para la próxima semana

- Completá la [guía de ejercicios del módulo 3](guia-ejercicios-modulo-3.ipynb) — **entrega: viernes 30/10/2026**.
- La próxima semana arranca el Módulo 4: **NumPy y Pandas** — todo lo que venimos haciendo a mano (medias, filtrados, columnas de CSV) en una línea, sobre miles o millones de datos. Ahí Python empieza a pagar en serio.